In [30]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import warnings

In [31]:

THRESHOLD = 0.5
Y_MIN, Y_MAX = 0.0, 1.0
SIZE = 32
MAX_LEN = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Labelling Regression score
def score_to_category(score):
    if score <= 0.20:   return "Safe"
    elif score <= 0.45: return "Ambiguous"
    elif score <= 0.75: return "Implicit/Covert Hate"
    else:               return "Explicit Toxicity"

In [33]:
df = pd.read_csv('../data/processed/train_cleaned.csv')
print(f"Loaded {len(df)} rows | target range: {df['target'].min():.4f}–{df['target'].max():.4f}")


df['category'] = df['target'].apply(score_to_category)
print(df['category'].value_counts())
df.head(5)

Loaded 31225 rows | target range: 0.0000–1.0000
category
Safe                    15618
Ambiguous                8899
Implicit/Covert Hate     4481
Explicit Toxicity        2227
Name: count, dtype: int64


,text,cleaned_text,target,toxic,severe_toxic,obscene,threat,insult,identity_hate,category
0,"""==dont leave==\nWikipedia needs people like y...",dont leave Wikipedia needs people like you Rem...,0.00,0,0,0,0,0,0,Safe
1,First problem ParacusForward like most wiki ed...,First problem ParacusForward like most wiki ed...,0.25,1,0,0,0,0,0,Ambiguous
2,"Grow up, cyber yuppie.",Grow up cyber yuppie,0.25,1,0,0,0,0,0,Ambiguous
3,"""\nThis is ridiculous. I can't respond BECAUS...",This is ridiculous I cant respond BECAUSE YOU ...,0.00,0,0,0,0,0,0,Safe
4,Here is proof of notability \nCalton has aimed...,Here is proof of notability Calton has aimed h...,0.25,1,0,0,0,0,0,Ambiguous


In [34]:
class ToxicityDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        self.texts    = texts.reset_index(drop=True)
        self.targets  = targets.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'target':         torch.tensor(self.targets[idx], dtype=torch.float)
        }

In [ ]:
class ToxicityModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        
        self.encoder = AutoModel.from_pretrained(
            model_name,
            attn_implementation="eager"
        )
        hidden = self.encoder.config.hidden_size

        self.regressor = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.identity_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           output_attentions=True)
        cls_emb    = out.last_hidden_state[:, 0, :]
        toxicity   = self.regressor(cls_emb).squeeze(-1)
        identity   = self.identity_classifier(cls_emb).squeeze(-1)
        attentions = out.attentions
        return toxicity, identity, attentions


def ear_penalty(attentions, lambda_ear=0.001):
    penalty = 0.0
    for layer_attn in attentions:
        attn = layer_attn.clamp(min=1e-9)
        entropy = -(attn * attn.log()).sum(-1)  
        penalty += entropy.mean()              
    penalty = penalty / len(attentions)         
    return -lambda_ear * penalty  

In [ ]:
class ToxicityModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        
        self.encoder = AutoModel.from_pretrained(
            model_name,
            attn_implementation="eager"
        )
        hidden = self.encoder.config.hidden_size

        self.regressor = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        self.identity_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           output_attentions=True)
        cls_emb    = out.last_hidden_state[:, 0, :]
        toxicity   = self.regressor(cls_emb).squeeze(-1)
        identity   = self.identity_classifier(cls_emb).squeeze(-1)
        attentions = out.attentions
        return toxicity, identity, attentions


def ear_penalty(attentions, lambda_ear=0.001):
    penalty = 0.0
    for layer_attn in attentions:
        attn = layer_attn.clamp(min=1e-9)
        entropy = -(attn * attn.log()).sum(-1)  
        penalty += entropy.mean()              
    penalty = penalty / len(attentions)         
    return -lambda_ear * penalty               

In [ ]:
ACCUMULATION_STEPS = 4
scaler = torch.cuda.amp.GradScaler()

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for i, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        targets   = batch['target'].to(device) 

        with torch.amp.autocast('cuda'):
            toxicity, identity, attentions = model(input_ids, attn_mask)
            loss_mse = weighted_mse(toxicity.float(), targets.float())
            loss_ear = ear_penalty(attentions)
            loss     = (loss_mse + loss_ear) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS

    return total_loss / len(loader)


def eval_epoch(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            targets   = batch['target'].to(device)

            with torch.amp.autocast('cuda'):
                toxicity, _, _ = model(input_ids, attn_mask)

            all_preds.extend(toxicity.cpu().float().numpy())
            all_targets.extend(targets.cpu().float().numpy())

    preds   = np.array(all_preds)
    targets = np.array(all_targets)
    mse     = mean_squared_error(targets, preds)

    bin_preds   = (preds   >= THRESHOLD).astype(int)
    bin_targets = (targets >= THRESHOLD).astype(int)
    precision = precision_score(bin_targets, bin_preds, zero_division=0)
    recall    = recall_score(bin_targets, bin_preds, zero_division=0)
    f1        = f1_score(bin_targets, bin_preds, zero_division=0)

    return mse, precision, recall, f1

/tmp/ipykernel_66857/3277931943.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [38]:
print(torch.cuda.get_device_name(0))
print(f"VRAM total:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free:   {torch.cuda.memory_reserved(0) / 1e9:.1f} GB reserved")

NVIDIA GeForce RTX 3050 Laptop GPU
VRAM total:  4.0 GB
VRAM free:   0.0 GB reserved


In [ ]:
print(DEVICE)  
print(torch.cuda.is_available())

cuda
True


In [ ]:
import gc
from torch.utils.data import WeightedRandomSampler

# Clear GPU memory
if 'model' in dir():
    del model
torch.cuda.empty_cache()
gc.collect()

lgbtq_neutral = pd.DataFrame({
    'cleaned_text': [
        "I'm a proud gay man",
        "My lesbian neighbor is very kind",
        "The bisexual community deserves respect",
        "She came out as lesbian last year",
        "He is gay and a great friend",
        "The gay couple moved in next door",
        "I support lesbian rights",
        "My bisexual friend is wonderful",
        "gay people deserve equal rights",
        "the lesbian couple adopted a child",
    ],
    'target': [0.02] * 10
})
lgbtq_neutral['category'] = 'Safe'
lgbtq_oversampled = pd.concat([lgbtq_neutral] * 100, ignore_index=True)
df_safe      = df[df['target'] <= 0.20].sample(
                   n=min(19000, (df['target'] <= 0.20).sum()), random_state=42)

df_ambiguous = df[(df['target'] > 0.20) & (df['target'] <= 0.45)].sample(
                   n=min(6000, ((df['target'] > 0.20) & (df['target'] <= 0.45)).sum()),
                   random_state=42)

df_implicit  = df[(df['target'] > 0.45) & (df['target'] <= 0.75)].sample(
                   n=min(4000, ((df['target'] > 0.45) & (df['target'] <= 0.75)).sum()),
                   random_state=42)

df_explicit  = df[df['target'] > 0.75].sample(
                   n=min(2200, (df['target'] > 0.75).sum()),
                   replace=(df['target'] > 0.75).sum() < 2200,
                   random_state=42)

df_clean = pd.concat([df_safe, df_ambiguous, df_implicit, df_explicit, lgbtq_oversampled]).sample(
               frac=1, random_state=42).reset_index(drop=True)
df_clean['category'] = df_clean['target'].apply(score_to_category)

print("Clean dataset distribution:")
print(df_clean['category'].value_counts())
print(f"Total: {len(df_clean)} rows")

MODEL_NAME = 'distilbert-base-uncased'

train_df, val_df = train_test_split(df_clean, test_size=0.15,
                                    random_state=42, stratify=df_clean['category'])

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = ToxicityDataset(train_df['cleaned_text'], train_df['target'], tokenizer, MAX_LEN)
val_ds   = ToxicityDataset(val_df['cleaned_text'],   val_df['target'],   tokenizer, MAX_LEN)


category_counts = train_df['category'].value_counts()
sample_weights  = train_df['category'].map(lambda c: 1.0 / category_counts[c]).values
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_ds),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=16,
                          num_workers=2, pin_memory=True)


def weighted_mse(preds, targets, threshold_implicit=0.45, threshold_explicit=0.75):
    weights = torch.ones_like(targets)
    weights = torch.where(targets >= threshold_implicit,
                          torch.full_like(targets, 5.0), weights)
    weights = torch.where(targets >= threshold_explicit,
                          torch.full_like(targets, 15.0), weights)
    return (weights * (preds - targets) ** 2).mean()


model     = ToxicityModel(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler    = torch.cuda.amp.GradScaler()


best_val_mse     = float('inf')
patience         = 5
patience_counter = 0
best_epoch       = 0

EPOCHS = 50
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, DEVICE)
    mse, prec, rec, f1 = eval_epoch(model, val_loader, DEVICE)
    scheduler.step()
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val MSE: {mse:.4f} | Precision: {prec:.4f} | "
          f"Recall: {rec:.4f} | F1: {f1:.4f}")

    if mse < best_val_mse:
        best_val_mse     = mse
        best_epoch       = epoch
        patience_counter = 0
        torch.save(model.state_dict(), '../models/distilbert_toxicity_best.pt')
        print(f"  New best saved (val MSE: {mse:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}. Best was epoch {best_epoch}.")
            break

model.load_state_dict(torch.load('../models/distilbert_toxicity_best.pt',
                                  map_location=DEVICE, weights_only=True))
print("Best model loaded.")

Clean dataset distribution:
category
Safe                    16618
Ambiguous                6000
Implicit/Covert Hate     4000
Explicit Toxicity        2200
Name: count, dtype: int64
Total: 28818 rows


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7810.62it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_66857/1116545496.py:88: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()


Epoch 1/50 | Train Loss: 0.1700 | Val MSE: 0.0470 | Precision: 0.6530 | Recall: 0.9108 | F1: 0.7607
  New best saved (val MSE: 0.0470)
Epoch 2/50 | Train Loss: 0.0874 | Val MSE: 0.0309 | Precision: 0.6935 | Recall: 0.8903 | F1: 0.7797
  New best saved (val MSE: 0.0309)
Epoch 3/50 | Train Loss: 0.0540 | Val MSE: 0.0279 | Precision: 0.7244 | Recall: 0.8505 | F1: 0.7824
  New best saved (val MSE: 0.0279)
Epoch 4/50 | Train Loss: 0.0393 | Val MSE: 0.0278 | Precision: 0.6829 | Recall: 0.8871 | F1: 0.7717
  New best saved (val MSE: 0.0278)
Epoch 5/50 | Train Loss: 0.0336 | Val MSE: 0.0256 | Precision: 0.7157 | Recall: 0.8688 | F1: 0.7848
  New best saved (val MSE: 0.0256)
Epoch 6/50 | Train Loss: 0.0326 | Val MSE: 0.0256 | Precision: 0.7157 | Recall: 0.8688 | F1: 0.7848
  No improvement (1/5)
Epoch 7/50 | Train Loss: 0.0302 | Val MSE: 0.0269 | Precision: 0.6955 | Recall: 0.8817 | F1: 0.7776
  No improvement (2/5)
Epoch 8/50 | Train Loss: 0.0289 | Val MSE: 0.0284 | Precision: 0.6709 | Recall:

In [41]:
def predict(text, model, tokenizer, device):
    model.eval()
    enc = tokenizer(text, return_tensors='pt', max_length=MAX_LEN,
                    truncation=True, padding='max_length')
    with torch.no_grad():
        score, _, _ = model(enc['input_ids'].to(device),
                            enc['attention_mask'].to(device))
    score = score.item()
    return {
        "raw_score":  score,
        "y_original": score * (Y_MAX - Y_MIN) + Y_MIN,
        "category":   score_to_category(score),
        "flag":       score >= THRESHOLD
    }



In [42]:
# Sanity Checks
tests = [
    # Explicit Toxicity (>0.75)
    ("eat shit fucko suck my asshole then die horribly i hope you get cancer",
     "expect >0.75 Explicit"),

    # Implicit/Covert Hate (0.46–0.75)
    ("I absolutely hate you, you disgusting piece of trash.",
     "expect 0.46–0.75 Implicit"),

    # Ambiguous (0.21–0.45)
    ("grandma terri die in trash if cking hate you",
     "expect 0.21–0.45 Ambiguous"),

    # Safe (<0.20) 
    ("I had a really wonderful day at the park today.",
     "expect <0.20 Safe"),

    # Safe (<0.20) identity term, neutral context
    ("My new neighbor is a Muslim man.",
     "expect <0.20 Safe"),

    # Safe (<0.20) negation
    ("I am not a hateful person, and I do not despise you.",
     "expect <0.20 Safe"),
]

for text, expectation in tests:
    r = predict(text, model, tokenizer, DEVICE)
    status = "✓" if (
        ("0.75" in expectation and r['raw_score'] > 0.75) or
        ("0.46" in expectation and 0.46 <= r['raw_score'] <= 0.75) or
        ("0.21" in expectation and 0.21 <= r['raw_score'] <= 0.45) or
        ("Safe" in expectation and r['raw_score'] < 0.20)
    ) else "✗"
    print(f"{status} [{r['category']:25s}] score={r['raw_score']:.3f} | {expectation}")
    print(f"  → {text[:70]}")

✓ [Explicit Toxicity        ] score=0.975 | expect >0.75 Explicit
  → eat shit fucko suck my asshole then die horribly i hope you get cancer
✓ [Implicit/Covert Hate     ] score=0.696 | expect 0.46–0.75 Implicit
  → I absolutely hate you, you disgusting piece of trash.
✗ [Implicit/Covert Hate     ] score=0.541 | expect 0.21–0.45 Ambiguous
  → grandma terri die in trash if cking hate you
✓ [Safe                     ] score=0.019 | expect <0.20 Safe
  → I had a really wonderful day at the park today.
✓ [Safe                     ] score=0.018 | expect <0.20 Safe
  → My new neighbor is a Muslim man.
✗ [Ambiguous                ] score=0.235 | expect <0.20 Safe
  → I am not a hateful person, and I do not despise you.


In [43]:
# Real world tests
examples = [
    'Honestly you should go fuck yourself',
    'You should just die of cancer',
    'Fucking whore',
    'You are the cutest being on earth',
    "I'm a proud Muslim",
    'That Muslim there is suspicious as hell',
    "Your Character is so strong",
    "Wow, Skott! I am genuinely impressed. I didn't think it was humanly possible to fit that much ego into one single suit, but look at you—breaking records every day!",
    'I love you in whole wide world',
    'That nigga is annoying',
    
]

for text in examples:
    result = predict(text, model, tokenizer, DEVICE)
    print(f"Text : {text} : Result :  {result['raw_score']}")


Text : Honestly you should go fuck yourself : Result :  0.5630452036857605
Text : You should just die of cancer : Result :  0.6022533774375916
Text : Fucking whore : Result :  0.8863688111305237
Text : You are the cutest being on earth : Result :  0.2803709805011749
Text : I'm a proud Muslim : Result :  0.04765259847044945
Text : That Muslim there is suspicious as hell : Result :  0.3438512086868286
Text : Your Character is so strong : Result :  0.033933937549591064
Text : Wow, Skott! I am genuinely impressed. I didn't think it was humanly possible to fit that much ego into one single suit, but look at you—breaking records every day! : Result :  0.07810401171445847
Text : I love you in whole wide world : Result :  0.04363551735877991
Text : That nigga is annoying : Result :  0.7668774724006653
